In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Data Quality

Before performing any analysis, it is essential to assess and improve the quality of the data. Real-world sensor data often contains missing values, duplicates, outliers, and gaps in the time series.

This chapter covers common techniques for detecting and handling these issues.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv"

df = pd.read_csv(url, sep=";")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")
df.head()

## Missing Values

Missing values (`NaN`) are common in sensor data due to communication failures, sensor outages, or maintenance periods.

In [ ]:
# Count missing values per column
df.isna().sum()

In [ ]:
# Percentage of missing values
missing_pct = (df.isna().sum() / len(df)) * 100
missing_pct.round(2)

In [ ]:
# Detailed info including non-null counts
df.info()

### Visualizing Missing Data

A heatmap of missing values helps identify patterns — for example, whether all sensors fail at the same time.

In [ ]:
# Resample to daily resolution for a cleaner visualization
missing_daily = df.isna().resample("D").mean() * 100

fig = px.imshow(
    missing_daily.T,
    labels=dict(x="Date", y="Column", color="% Missing"),
    title="Missing Data Pattern (% per day)",
    color_continuous_scale="Reds",
    aspect="auto"
)
fig.show()

## Detecting Gaps in Time Series

Even if there are no `NaN` values, there might be missing timestamps — periods where no data was recorded at all.

In [ ]:
# Check the expected vs actual number of timestamps
expected_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="h")
print(f"Expected timestamps: {len(expected_range)}")
print(f"Actual timestamps:   {len(df)}")
print(f"Missing timestamps:  {len(expected_range) - len(df)}")

In [ ]:
# Find the missing timestamps
missing_timestamps = expected_range.difference(df.index)
print(f"Number of missing timestamps: {len(missing_timestamps)}")
if len(missing_timestamps) > 0:
    print(f"First missing: {missing_timestamps[0]}")
    print(f"Last missing:  {missing_timestamps[-1]}")

In [ ]:
# Detect gaps by checking time differences between consecutive rows
time_diffs = df.index.to_series().diff()
gaps = time_diffs[time_diffs > pd.Timedelta(hours=1)]
print(f"Number of gaps: {len(gaps)}")
gaps.head(10)

## Duplicates

Duplicate rows can occur when data is merged from multiple sources or exported incorrectly.

In [ ]:
# Check for duplicate rows
print(f"Number of duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Check for duplicate index values (duplicate timestamps)
print(f"Number of duplicate timestamps: {df.index.duplicated().sum()}")

In [ ]:
# Remove duplicates (if any)
df_clean = df[~df.index.duplicated(keep="first")]
print(f"Rows before: {len(df)}, after: {len(df_clean)}")

```{tip}
Use `keep="first"` to keep the first occurrence, `keep="last"` to keep the last, or `keep=False` to remove all duplicates.
```

## Outlier Detection

Outliers can result from sensor malfunctions, calibration errors, or genuine extreme events. Two common detection methods are the IQR method and the z-score method.

### IQR Method

The Interquartile Range (IQR) method flags values that fall below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.

In [ ]:
col = "FlatA_Temp"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

outliers_iqr = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
print(f"Number of outliers (IQR): {len(outliers_iqr)}")

In [ ]:
# Visualize outliers
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df[col],
    mode="lines", name=col, line=dict(color="steelblue")
))

fig.add_trace(go.Scatter(
    x=outliers_iqr.index, y=outliers_iqr[col],
    mode="markers", name="Outliers (IQR)",
    marker=dict(color="red", size=5)
))

fig.add_hline(y=upper_bound, line_dash="dash", line_color="red", annotation_text="Upper bound")
fig.add_hline(y=lower_bound, line_dash="dash", line_color="red", annotation_text="Lower bound")

fig.update_layout(title=f"Outlier Detection (IQR Method) — {col}", xaxis_title="Time", yaxis_title="Temperature [°C]")
fig.show()

### Z-Score Method

The z-score measures how many standard deviations a value is from the mean. Values with |z| > 3 are typically considered outliers.

In [ ]:
col = "FlatA_Temp"

mean = df[col].mean()
std = df[col].std()

df_temp = df.copy()
df_temp["z_score"] = (df[col] - mean) / std

outliers_z = df_temp[df_temp["z_score"].abs() > 3]
print(f"Number of outliers (z-score > 3): {len(outliers_z)}")
outliers_z[[col, "z_score"]].head(10)

## Interpolation and Filling Missing Values

Once missing values are identified, they can be filled using various strategies.

### Linear Interpolation

In [ ]:
col = "FlatA_Temp"

# Original data with NaN values
print(f"NaN values before: {df[col].isna().sum()}")

# Interpolate linearly
df_interp = df.copy()
df_interp[col] = df_interp[col].interpolate(method="linear")

print(f"NaN values after:  {df_interp[col].isna().sum()}")

In [ ]:
# Visualize a section with interpolated values
# Find a region with NaN values
nan_mask = df[col].isna()
if nan_mask.any():
    first_nan = df[nan_mask].index[0]
    start = first_nan - pd.Timedelta(days=2)
    end = first_nan + pd.Timedelta(days=2)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df.loc[start:end].index, y=df.loc[start:end, col],
        mode="lines+markers", name="Original", line=dict(color="steelblue")
    ))
    fig.add_trace(go.Scatter(
        x=df_interp.loc[start:end].index, y=df_interp.loc[start:end, col],
        mode="lines", name="Interpolated", line=dict(color="orange", dash="dash")
    ))
    fig.update_layout(title="Linear Interpolation of Missing Values", xaxis_title="Time", yaxis_title="Temperature [°C]")
    fig.show()

### Forward and Backward Fill

In [ ]:
# Forward fill: carry the last valid observation forward
df_ffill = df.copy()
df_ffill[col] = df_ffill[col].fillna(method="ffill")

# Backward fill: carry the next valid observation backward
df_bfill = df.copy()
df_bfill[col] = df_bfill[col].fillna(method="bfill")

print(f"Forward fill NaN remaining: {df_ffill[col].isna().sum()}")
print(f"Backward fill NaN remaining: {df_bfill[col].isna().sum()}")

### Fill with a Constant

In [ ]:
# Fill with the column mean
df_mean_fill = df.copy()
df_mean_fill[col] = df_mean_fill[col].fillna(df[col].mean())

print(f"NaN remaining: {df_mean_fill[col].isna().sum()}")

```{note}
Choose the imputation method carefully based on the nature of the data. Linear interpolation works well for slowly changing sensor data. Forward fill is appropriate when the last known value is the best estimate. Filling with the mean is generally not recommended for time series data.
```

## Summary Statistics for Quality Assessment

A quick quality overview can be compiled by combining several checks.

In [ ]:
quality_summary = pd.DataFrame({
    "total_rows": len(df),
    "non_null": df.count(),
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().sum() / len(df) * 100).round(2),
    "min": df.min(),
    "max": df.max(),
    "mean": df.mean().round(2),
    "std": df.std().round(2)
})

quality_summary

In [ ]:
# Visualize data completeness
completeness = (1 - df.isna().mean()) * 100

fig = px.bar(
    x=completeness.index,
    y=completeness.values,
    labels={"x": "Column", "y": "Completeness (%)"},
    title="Data Completeness per Column"
)
fig.update_layout(yaxis_range=[0, 105])
fig.show()